In [1]:
import sys, os, math, time, csv
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_util import make_vec_env
import shooter

In [2]:
def wrap_angle(angle: float) -> float:
    return (angle + math.pi) % (2.0 * math.pi) - math.pi

In [3]:
class FullRewardWrapper(gym.Wrapper):
    def __init__(self, env, failure_states, success_obs, success_act,
                 penalty_threshold=0.3, sil_threshold=0.1, sil_bonus=0.2,
                 repeat_angle_threshold=0.10, repeat_step_threshold=8,
                 repeat_penalty=3.0, tree_penalty=10.0):
        super().__init__(env)
        self.failure_states = np.array(failure_states)
        self.success_obs = np.array(success_obs)
        self.success_act = np.array(success_act).astype(int)
        self.penalty_threshold = penalty_threshold
        self.sil_threshold = sil_threshold
        self.sil_bonus = sil_bonus
        self.repeat_angle_threshold = repeat_angle_threshold   # rad
        self.repeat_step_threshold = repeat_step_threshold     # ticks
        self.repeat_penalty = repeat_penalty
        self.tree_penalty = tree_penalty
        self.prev_score = 0
        self.kills = 0
        self.current_step = 0

        
        self.last_shot_yaw = None
        self.last_shot_pitch = None
        self.last_shot_step = -100

        self._trees = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev_score = info["hunterScore"]
        self.kills = 0
        self.current_step = 0
        self.last_shot_yaw = None
        self.last_shot_pitch = None
        self.last_shot_step = -100
        try:
            self._trees = self.env.unwrapped._engine.state["trees"]
        except:
            self._trees = []
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.current_step += 1
        shaped = reward

        score_delta = info["hunterScore"] - self.prev_score
        self.prev_score = info["hunterScore"]

        # --- Kill bonus ---
        if score_delta > 0:
            shaped += score_delta * 5.0
            self.kills += 1

        # --- Aim bonus ---
        aim_err = self._calc_aim_error(obs)
        if aim_err is not None:
            shaped += 2.0 * math.exp(-aim_err**2 / 0.1)
            if aim_err < 0.04:
                shaped += 5.0
            if action == 1:
                if aim_err < 0.15:
                    shaped += 1.0
                else:
                    shaped -= 1.5

        # --- Tree penalty ---
        if action == 1 and self._is_tree_blocked(obs):
            shaped -= self.tree_penalty   

        # --- Wasteful shot penalty ---
        if action == 1 and self._count_alive(obs) == 0:
            shaped -= 5.0

        # --- Repeat Shot Penalty ---
        if action == 1:
            cur_yaw = obs[3] * math.pi
            cur_pitch = obs[4] * 0.9
            if self.last_shot_yaw is not None:
                yaw_diff = abs(wrap_angle(cur_yaw - self.last_shot_yaw))
                pitch_diff = abs(cur_pitch - self.last_shot_pitch)
                
                if (yaw_diff < self.repeat_angle_threshold and
                    pitch_diff < self.repeat_angle_threshold and
                    self.current_step - self.last_shot_step <= self.repeat_step_threshold):
                    
                    if score_delta <= 0:   
                        shaped -= self.repeat_penalty
            
            self.last_shot_yaw = cur_yaw
            self.last_shot_pitch = cur_pitch
            self.last_shot_step = self.current_step

        # --- Failure penalty ---
        if len(self.failure_states) > 0:
            dists = np.linalg.norm(self.failure_states - obs[:169], axis=1)
            if np.min(dists) < self.penalty_threshold:
                shaped -= 1.0

        # --- SIL bonus ---
        if len(self.success_obs) > 0:
            dists = np.linalg.norm(self.success_obs - obs[:169], axis=1)
            close_idx = np.where(dists < self.sil_threshold)[0]
            if len(close_idx) > 0 and np.any(self.success_act[close_idx] == action):
                shaped += self.sil_bonus

        info["kills"] = self.kills
        return obs, shaped, terminated, truncated, info

    # =============================================
    def _calc_aim_error(self, obs):
        
        best_dist = 1e9
        best_x = best_y = best_z = None
        for i in range(15):
            j = 6 + i*8
            if obs[j+6] > 0.5:  # alive
                x = obs[j] * 100.0
                z = obs[j+1] * 100.0
                y = obs[j+5] * 20.0
                d = math.sqrt(x*x + z*z)
                if d < best_dist:
                    best_dist = d
                    best_x, best_y, best_z = x, y, z
        if best_x is None:
            return None
        cur_yaw = obs[3] * math.pi
        cur_pitch = obs[4] * 0.9
        ideal_yaw = math.atan2(best_z, best_x)
        xz_dist = max(math.sqrt(best_x**2 + best_z**2), 0.5)
        ideal_pitch = math.atan2(best_y - 2.3, xz_dist)
        yaw_err = abs(wrap_angle(ideal_yaw - cur_yaw))
        pitch_err = abs(ideal_pitch - cur_pitch)
        return yaw_err + pitch_err

    def _is_tree_blocked(self, obs):
        
        if not self._trees:
            return False
        yaw = obs[3] * math.pi
        pitch = obs[4] * 0.9
        cp = math.cos(pitch)
        dx = math.cos(yaw) * cp
        dy = math.sin(pitch)
        dz = math.sin(yaw) * cp
        bx, by, bz = dx*3.0, 2.3 + dy*3.0, dz*3.0
        d = 0.0
        while d < 90.0:
            bx += dx * 3.0
            by += dy * 3.0
            bz += dz * 3.0
            d += 3.0
            if by < -2.0 or by > 80.0:
                break
            for t in self._trees:
                tx, tz, tr = t["x"], t["z"], t.get("r", 2.5)
                if (bx - tx)**2 + (bz - tz)**2 < (tr + 0.3)**2:
                    return True
        return False

    def _count_alive(self, obs):
        return sum(1 for i in range(15) if obs[6+i*8+6] > 0.5)

In [4]:
from torch.utils.tensorboard import SummaryWriter
import time

class DetailedLogCallbackWithTB(BaseCallback):
    def __init__(self, writer, log_every=100):
        super().__init__()
        self.writer = writer
        self.log_every = log_every
        self.ep_count = 0
        self.cur_rewards = {}
        self.ep_rewards = []
        self.ep_kills = []
        self.ep_scores = []
        self.start_time = time.time()

    def _on_step(self):
        infos = self.locals.get("infos", [{}] * len(self.locals["dones"]))
        for i, done in enumerate(self.locals["dones"]):
            self.cur_rewards[i] = self.cur_rewards.get(i, 0) + self.locals["rewards"][i]
            if done:
                self.ep_count += 1
                self.ep_rewards.append(self.cur_rewards[i])
                self.ep_kills.append(infos[i].get("kills", 0))
                self.ep_scores.append(infos[i].get("hunterScore", 0))
                self.cur_rewards[i] = 0

                if self.ep_count % self.log_every == 0:
                    recent_r = self.ep_rewards[-self.log_every:]
                    recent_k = self.ep_kills[-self.log_every:]
                    recent_s = self.ep_scores[-self.log_every:]
                    elapsed = time.time() - self.start_time
                    mean_r = np.mean(recent_r)
                    max_r = np.max(recent_r)
                    min_r = np.min(recent_r)
                    sum_k = int(np.sum(recent_k))
                    mean_s = np.mean(recent_s)

                   
                    print(f"Ep {self.ep_count:5d} | "
                          f"Mean: {mean_r:7.2f} | Max: {max_r:7.2f} | Min: {min_r:7.2f} | "
                          f"Kill: {sum_k:4d} | Score: {mean_s:6.1f} | "
                          f"Step {self.num_timesteps:7d} | Time: {elapsed/60:6.1f}m")

                    
                    if self.writer:
                        self.writer.add_scalar("Episode/Mean_Reward", mean_r, self.ep_count)
                        self.writer.add_scalar("Episode/Mean_Score", mean_s, self.ep_count)
                        self.writer.add_scalar("Episode/Sum_Kills", sum_k, self.ep_count)
                        self.writer.add_scalar("Training/Timesteps", self.num_timesteps, self.ep_count)
        return True

In [5]:
failure_states = np.load("failure_states_v2.npy")
success_obs = np.load("success_obs.npy")
success_act = np.load("success_act.npy")

writer = SummaryWriter(log_dir="runs/BCPPO_selfCollect")
callback = DetailedLogCallbackWithTB(writer=writer, log_every=10)

def make_env():
    env = gym.make("Shooter-v0", render_mode=None)
    return FullRewardWrapper(env, failure_states, success_obs, success_act,
                             penalty_threshold=0.3, sil_threshold=0.1, sil_bonus=0.2)

vec_env = make_vec_env(make_env, n_envs=4)

model = PPO.load("ppo_v9_sil_failure", env=vec_env, device="cuda")


for param_group in model.policy.optimizer.param_groups:
    param_group['lr'] = 1e-5


model.learn(total_timesteps=200_000, callback=callback, progress_bar=True)

writer.close()
model.save("ppo_v9_sil_failure")

Output()

/home/jupyter-st125843/.local/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Ep    10 | Mean: 2961.37 | Max: 8632.97 | Min:   70.45 | Kill:  183 | Score:  193.6 | Step   17208 | Time:    3.5m

Ep    20 | Mean: 4601.23 | Max: 13238.28 | Min:  222.04 | Kill:  281 | Score:  324.8 | Step   28628 | Time:    5.7m

Ep    30 | Mean: 6344.03 | Max: 12219.02 | Min:  318.37 | Kill:  381 | Score:  466.8 | Step   45280 | Time:    9.2m

...

Ep    80 | Mean: 6857.32 | Max: 10089.40 | Min:  347.75 | Kill:  406 | Score:  529.0 | Step  135864 | Time:   29.6m

Ep    90 | Mean: 3942.02 | Max: 9125.77 | Min:  176.35 | Kill:  235 | Score:  302.2 | Step  148808 | Time:   32.5m

Ep   100 | Mean: 5004.48 | Max: 11296.28 | Min:  178.60 | Kill:  282 | Score:  342.2 | Step  163488 | Time:   35.7m

Ep   110 | Mean: 6001.83 | Max: 13369.30 | Min:  335.72 | Kill:  337 | Score:  457.6 | Step  177896 | Time:   39.0m

Ep   120 | Mean: 3944.32 | Max: 9723.42 | Min:  199.81 | Kill:  224 | Score:  307.6 | Step  193052 | Time:   42.4m